In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
DATA_PATH = '/home/ducminh/Desktop/Academic_Study/Self-learning/Code/StoicCoding/Others/Data/creditcard.csv'
convert_label = lambda x: float(x.strip('"') or  -1)

dataset = np.genfromtxt(DATA_PATH,
                        delimiter=',',
                        skip_header=1,
                        converters= {-1: convert_label},
                        dtype=None,
                        encoding=None)
dataset[:10, :]

array([[ 0.00000000e+00, -1.35980713e+00, -7.27811733e-02,
         2.53634674e+00,  1.37815522e+00, -3.38320770e-01,
         4.62387778e-01,  2.39598554e-01,  9.86979013e-02,
         3.63786970e-01,  9.07941720e-02, -5.51599533e-01,
        -6.17800856e-01, -9.91389847e-01, -3.11169354e-01,
         1.46817697e+00, -4.70400525e-01,  2.07971242e-01,
         2.57905802e-02,  4.03992960e-01,  2.51412098e-01,
        -1.83067779e-02,  2.77837576e-01, -1.10473910e-01,
         6.69280749e-02,  1.28539358e-01, -1.89114844e-01,
         1.33558377e-01, -2.10530535e-02,  1.49620000e+02,
         0.00000000e+00],
       [ 0.00000000e+00,  1.19185711e+00,  2.66150712e-01,
         1.66480113e-01,  4.48154078e-01,  6.00176493e-02,
        -8.23608088e-02, -7.88029833e-02,  8.51016549e-02,
        -2.55425128e-01, -1.66974414e-01,  1.61272666e+00,
         1.06523531e+00,  4.89095016e-01, -1.43772296e-01,
         6.35558093e-01,  4.63917041e-01, -1.14804663e-01,
        -1.83361270e-01, -1.45

In [3]:
n_samples = dataset.shape[0] 
n_features = dataset.shape[1] - 1
n_classes = len(np.unique(dataset[:, :-1]))

In [ ]:
np.random.seed(1)
dataset = np.random.permutation(dataset)

X = dataset[:, :-1].astype(np.float64)

# Start Encoding
y_idx = dataset[:, -1].astype(np.int32)
y = np.array([np.zeros(n_classes) for _ in range(y_idx.shape[0])])
y[np.arange(len(y_idx)), y_idx] = 1

In [ ]:
# Normalization with min-max normalization
X_min = np.min(X, axis=0)
X_max = np.max(X, axis=0)

X = (X - X_min)/(X_max - X_min)

X = np.c_[np.ones((X.shape[0], 1)), X]


In [ ]:
# Distributing the dataset for training, testing and validating
TRAIN_SIZE = 0.7
VAL_SIZE = 0.2

TRAIN_IDX_END = int(TRAIN_SIZE * dataset.shape[0])
VAL_IDX_END = int(TRAIN_IDX_END + (VAL_SIZE * dataset.shape[0]))

X_train, y_train = X[:TRAIN_IDX_END], y[:TRAIN_IDX_END]
X_val, y_val = X[TRAIN_IDX_END:VAL_IDX_END], y[TRAIN_IDX_END:VAL_IDX_END]
X_test, y_test = X[VAL_IDX_END:], y[VAL_IDX_END:]

In [16]:
def softmax(z):
    numerator = np.exp(z)
    denominator = np.sum([np.exp(i) for i in z])
    return numerator/denominator

arr = np.array(np.random.rand(2, 3))
print(arr)
print(softmax(arr))
print(np.sum(softmax(arr)))

[[0.83745713 0.38034741 0.26778606]
 [0.90602471 0.84920689 0.37573505]]
[[0.20359086 0.12889559 0.11517369]
 [0.21804031 0.20599711 0.12830244]]
1.0


In [29]:
def predict(X, theta):
    z = X.dot(theta)
    return softmax(z)

X = np.array(np.random.rand(2, 3))
print(X.shape)
theta = np.array([1, 3, 10]).reshape(3, 1)
print(X.dot(theta))
predict(X, theta)

(2, 3)
[[6.6023606 ]
 [3.10270409]]


array([[0.97067799],
       [0.02932201]])

In [ ]:
def loss(y_hat, y):
    result = -np.log(y.T.dot(y_hat))
    return result

In [ ]:
def gradient(X, y, y_hat):
    grad = X.T.dot(y_hat - y)
    return grad

In [ ]:
def fit(X_train, y_train, theta, epochs, learning_rate):
    losses = []

    for epoch in range(epochs):
        process_bar = tqdm(range(n_samples), 
                        desc=f"EPOCH {epoch}", position=0)
        for i in process_bar:
            X_i = X_train[i]
            y_i = y_train[i]

            X_i = X_i.reshape((n_features, 1))
            y_i = y_i.reshape((n_classes, 1))

            y_hat = predict(X_i, theta)

            train_loss = loss(y_hat, y)[0][0]
            losses.append(train_loss)

            gradient = gradient(X_i, y_i, theta)
            theta = theta - learning_rate*gradient

            process_bar.postfix({"Train Loss": train_loss})

    print("\nTraining Complete")
    return theta, losses

In [ ]:
def evaluate(X, y, theta):
    y_hat = predict(X, theta)
    loss = loss(y_hat, y)
    accuracy = np.sum(y_hat == y)

    return (loss/n_samples), (accuracy/n_samples)

In [ ]:
# Training Model

theta = np.random.uniform(X_train.shape[0], np.unique(y_train, axis=0).reshape[0])

EPOCHS = 1
learning_rate = 1e-3

theta, losses = fit(X_train, y_train, theta, EPOCHS, learning_rate)

In [ ]:
# Evaluate the Model
val_loss, val_acc = evaluate(X_val, y_val, theta)
print("Validation loss: ", np.round(val_loss, 3))
print("Validation accuracy: ", np.round(val_acc, 3))


test_loss, test_acc = evaluate(X_test, y_test, theta)
print("Test loss: ", np.round(test_loss, 3))
print("Test accuracy: ", np.round(test_acc, 3))